# Manual review: resolving flagged accusation targets

Walks through `flagged_for_review.csv` one row at a time. For each row you can:

- type one or more player names, comma-separated (e.g. `Elliot, Sian`) → resolves the target and writes it directly into the annotation JSON
- press Enter (empty input) or type `u` → **confirms** the target is genuinely unresolvable; stays `UNKNOWN`, but is marked as manually reviewed so you know it's been looked at, not just skipped
- type `s` → skip this row for now (comes back next time you run the loop)
- type `q` → stop the loop early; everything done so far is already saved

Have the source transcript open in another window (`ready_for_annotation/<source>/<...>.txt`) — use the line number printed for each row to jump to it.

Progress is saved after every single answer (both to the CSV and to the annotation JSON), so it's safe to stop and resume anytime.


In [1]:
import json
import csv
import os
from pathlib import Path

# ---- adjust these if your paths differ ----
CSV_PATH = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\accusation_transcripts\acc_targets\flagged_for_review.csv"
)
OUTPUT_ROOT = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\accusation_transcripts\acc_targets"
)
# --------------------------------------------

FIELDNAMES_EXTRA = ["review_status", "resolved_accused"]


def load_rows():
    with CSV_PATH.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    # add tracking columns if this is the first time we're running the notebook
    for row in rows:
        row.setdefault("review_status", "")       # "", "resolved", "confirmed_unknown", "skipped"
        row.setdefault("resolved_accused", "")
    return rows


def save_rows(rows):
    fieldnames = list(rows[0].keys()) if rows else []
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def update_annotation_json(row, new_accused):
    """Find the matching relation inside the game's output JSON and update it.
    new_accused: list[str] to resolve to, or None to just mark as manually
    confirmed while leaving accused=["UNKNOWN"] as is.
    """
    json_path = OUTPUT_ROOT / row["output_file"].replace(chr(92), os.sep)
    record = json.loads(json_path.read_text(encoding="utf-8"))

    target_line = int(row["line_number"])
    target_type = row["type"]
    target_evidence = row["evidence"]

    for item in record.get("items", []):
        if item.get("line_number") != target_line:
            continue
        for relation in item.get("relations", []):
            if relation.get("type") != target_type:
                continue
            if relation.get("accused") != ["UNKNOWN"]:
                continue
            # evidence as a tiebreaker in the rare case of duplicate type+UNKNOWN on one line
            if relation.get("evidence") != target_evidence:
                continue

            relation["manually_reviewed"] = True
            if new_accused is not None:
                relation["accused"] = new_accused
            else:
                relation["confirmed_unresolvable"] = True

            # recompute requires_review: true only if some relation on this
            # item is still an unresolved UNKNOWN
            item["requires_review"] = any(
                r.get("accused") == ["UNKNOWN"] and not r.get("confirmed_unresolvable")
                for r in item.get("relations", [])
            )

            json_path.write_text(
                json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            return True

    print(f"  WARNING: could not find matching relation in {json_path} "
          f"(line {target_line}, type {target_type}) -- nothing was updated.")
    return False


rows = load_rows()
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(rows)} total flagged rows, {len(pending)} still pending review.")


74 total flagged rows, 74 still pending review.


In [2]:
for row in pending:
    print("=" * 70)
    print(f"File     : {row['output_file']}")
    print(f"Game     : {row['game']}   (session: {row['session']})")
    print(f"Line     : {row['line_number']}")
    print(f"Accuser  : {row['accuser']}")
    print(f"Type     : {row['type']}")
    print(f"Evidence : {row['evidence']}")
    print("-" * 70)

    answer = input(
        "Names (comma-separated) / Enter or 'u' = confirm unresolvable / "
        "'s' = skip / 'q' = quit: "
    ).strip()

    if answer.lower() == "q":
        print("Stopping. Progress so far is saved.")
        break

    if answer.lower() == "s":
        row["review_status"] = "skipped"
        save_rows(rows)
        continue

    if answer == "" or answer.lower() == "u":
        update_annotation_json(row, new_accused=None)
        row["review_status"] = "confirmed_unknown"
        save_rows(rows)
        print("-> confirmed unresolvable.")
        continue

    names = [n.strip() for n in answer.split(",") if n.strip()]
    ok = update_annotation_json(row, new_accused=names)
    if ok:
        row["review_status"] = "resolved"
        row["resolved_accused"] = ", ".join(names)
        save_rows(rows)
        print(f"-> resolved to {names}.")
    else:
        print("-> NOT saved (see warning above). Row left pending -- try again.")

print("=" * 70)
remaining = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"Done for now. {len(remaining)} rows still pending "
      f"(skipped ones will show up again next run).")


File     : Ego4D\0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0_Game1.json
Game     : Game1   (session: 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0)
Line     : 136
Accuser  : Jack
Type     : werewolf
Evidence : Maybe he's just living his best Werewolf life out here.
----------------------------------------------------------------------
-> confirmed unresolvable.
File     : Ego4D\0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0_Game2.json
Game     : Game2   (session: 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0)
Line     : 157
Accuser  : Brent
Type     : werewolf
Evidence : if one of my two theories about you guys potentially being sus or Jack potentially being sus is true
----------------------------------------------------------------------
-> resolved to ['Jack'].
File     : Ego4D\0c2659db-7bd4-4b37-9b08-4e247befe382_Game4.json
Game     : Game4   (session: 0c2659db-7bd4-4b37-9b08-4e247befe382)
Line     : 163
Accuser  : James
Type     : werewolf
Evidence : I still think it's her
------------------------------------------

Stopped partway, or came back later? Re-run the config cell above (it reloads the CSV with your saved progress), then run the cell below to rebuild `pending`, then re-run the loop cell.

In [ ]:
# Re-run this cell (instead of the one above) to only go through rows you
# explicitly skipped last time, without re-listing already-resolved ones.
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(pending)} rows pending (including previously skipped).")
